# Historical analysis notebook
Code-only copy. Outputs and original narrative cells were removed. See docs/reproduction.md for the revised analysis and execution order.


In [ ]:
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
def locate_project_root() -> Path:
    start = Path.cwd().resolve()
    for root in [start, *start.parents]:
        if (root / 'results').exists() and (root / 'images').exists():
            return root
    raise FileNotFoundError('Unable to locate project root')

PROJECT_ROOT = locate_project_root()
RESULTS_DIR = PROJECT_ROOT / 'results'
IMAGES_DIR = PROJECT_ROOT / 'images'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

GROUPS = ('all_signals', 'no_gas_exchange', 'hemo_plus_nirs', 'nirs')
MODELS = ('Ridge', 'ElasticNet', 'PLS')
GROUP_DISPLAY = {
    'all_signals': 'All signals',
    'no_gas_exchange': 'No gas exchange',
    'hemo_plus_nirs': 'Hemodynamics + NIRS',
    'nirs': 'NIRS only',
}

PROJECT_ROOT

In [ ]:
def safe_name(value: str) -> str:
    return value.replace(' ', '_').replace('/', '_')

def detect_column(df: pd.DataFrame, candidates: List[str]) -> str:
    lower_map = {col.lower(): col for col in df.columns}
    for candidate in candidates:
        key = candidate.lower()
        if key in lower_map:
            return lower_map[key]
    raise KeyError(f'Could not find candidate columns {candidates}')

def load_prediction_table(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)

    df = pd.read_csv(path)
    sid_col = detect_column(df, ['subject_id', 'Subject', 'ID'])
    actual_col = detect_column(df, ['actual_vo2peak', 'actual', 'y_true', 'VO2peak_abs'])
    pred_col = detect_column(df, ['predicted_vo2peak', 'predicted', 'y_pred', 'prediction'])

    out = df[[sid_col, actual_col, pred_col]].copy()
    out = out.rename(columns={sid_col: 'subject_id', actual_col: 'actual', pred_col: 'predicted'})
    out = out.dropna(subset=['subject_id', 'actual', 'predicted']).copy()
    out['subject_id'] = out['subject_id'].astype(str)
    return out.set_index('subject_id').sort_index()

def safe_r2(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    if np.isclose(ss_tot, 0.0):
        return np.nan
    return float(1.0 - ss_res / ss_tot)

def compute_delta_metrics(actual_delta: np.ndarray, pred_delta: np.ndarray) -> Dict[str, float]:
    delta_error = pred_delta - actual_delta
    abs_delta_error = np.abs(delta_error)

    out: Dict[str, float] = {
        'MAE_delta': float(np.mean(abs_delta_error)),
        'RMSE_delta': float(np.sqrt(np.mean(delta_error ** 2))),
        'R2_delta': safe_r2(actual_delta, pred_delta),
        'MeanDeltaError': float(np.mean(delta_error)),
        'StdDeltaError': float(np.std(delta_error, ddof=1)) if len(delta_error) > 1 else np.nan,
    }

    if len(actual_delta) >= 2 and np.std(actual_delta, ddof=1) > 0 and np.std(pred_delta, ddof=1) > 0:
        pearson_r, pearson_p = stats.pearsonr(pred_delta, actual_delta)
    else:
        pearson_r, pearson_p = np.nan, np.nan

    if len(actual_delta) >= 2:
        spearman_rho, spearman_p = stats.spearmanr(pred_delta, actual_delta)
    else:
        spearman_rho, spearman_p = np.nan, np.nan

    if len(actual_delta) >= 2 and np.std(pred_delta, ddof=1) > 0:
        calib = stats.linregress(pred_delta, actual_delta)
        cal_slope, cal_intercept = float(calib.slope), float(calib.intercept)
    else:
        cal_slope, cal_intercept = np.nan, np.nan

    if len(actual_delta) >= 2 and not np.allclose(delta_error, 0.0):
        t_stat, t_p = stats.ttest_rel(pred_delta, actual_delta)
        try:
            w_stat, w_p = stats.wilcoxon(pred_delta, actual_delta, zero_method='wilcox', alternative='two-sided')
        except ValueError:
            w_stat, w_p = np.nan, np.nan
        sd_err = np.std(delta_error, ddof=1)
        dz = float(np.mean(delta_error) / sd_err) if sd_err > 0 else np.nan
    else:
        t_stat, t_p, w_stat, w_p, dz = np.nan, np.nan, np.nan, np.nan, np.nan

    out.update({
        'Pearson_r': float(pearson_r) if not np.isnan(pearson_r) else np.nan,
        'Pearson_p': float(pearson_p) if not np.isnan(pearson_p) else np.nan,
        'Spearman_rho': float(spearman_rho) if not np.isnan(spearman_rho) else np.nan,
        'Spearman_p': float(spearman_p) if not np.isnan(spearman_p) else np.nan,
        'Calibration_slope': cal_slope,
        'Calibration_intercept': cal_intercept,
        'PairedT_stat': float(t_stat) if not np.isnan(t_stat) else np.nan,
        'PairedT_p': float(t_p) if not np.isnan(t_p) else np.nan,
        'Wilcoxon_stat': float(w_stat) if not np.isnan(w_stat) else np.nan,
        'Wilcoxon_p': float(w_p) if not np.isnan(w_p) else np.nan,
        'Cohens_dz': dz,
    })
    return out

In [ ]:
summary_records: List[Dict[str, float]] = []
subject_records: List[Dict[str, float]] = []

for group in GROUPS:
    for model in MODELS:
        pre_path = RESULTS_DIR / f"nested_{safe_name(group)}_{safe_name(model)}_loso_predictions.csv"
        post_path = RESULTS_DIR / f"e1_zero_shot_{safe_name(group)}_{safe_name(model)}_predictions.csv"

        pre_df = load_prediction_table(pre_path)
        post_df = load_prediction_table(post_path)

        common_subjects = sorted(set(pre_df.index).intersection(post_df.index))
        if not common_subjects:
            continue

        pre_common = pre_df.loc[common_subjects].copy()
        post_common = post_df.loc[common_subjects].copy()

        paired = pd.DataFrame(index=common_subjects)
        paired['actual_pre'] = pre_common['actual']
        paired['pred_pre'] = pre_common['predicted']
        paired['actual_post'] = post_common['actual']
        paired['pred_post'] = post_common['predicted']
        paired['actual_delta'] = paired['actual_post'] - paired['actual_pre']
        paired['pred_delta'] = paired['pred_post'] - paired['pred_pre']
        paired['delta_error'] = paired['pred_delta'] - paired['actual_delta']
        paired['abs_delta_error'] = paired['delta_error'].abs()
        paired['sign_match'] = np.sign(paired['pred_delta']) == np.sign(paired['actual_delta'])

        actual_delta = paired['actual_delta'].to_numpy(dtype=float)
        pred_delta = paired['pred_delta'].to_numpy(dtype=float)
        metric_dict = compute_delta_metrics(actual_delta, pred_delta)

        sign_match_n = int(paired['sign_match'].sum())
        sign_match_pct = float(100.0 * sign_match_n / len(paired))

        summary_records.append({
            'Group': group,
            'GroupLabel': GROUP_DISPLAY[group],
            'Model': model,
            'NCommonSubjects': int(len(paired)),
            'ActualDelta_mean': float(paired['actual_delta'].mean()),
            'ActualDelta_sd': float(paired['actual_delta'].std(ddof=1)) if len(paired) > 1 else np.nan,
            'PredDelta_mean': float(paired['pred_delta'].mean()),
            'PredDelta_sd': float(paired['pred_delta'].std(ddof=1)) if len(paired) > 1 else np.nan,
            'SignAgreementN': sign_match_n,
            'SignAgreementPct': sign_match_pct,
            **metric_dict,
        })

        for sid, row in paired.iterrows():
            subject_records.append({
                'subject_id': sid,
                'Group': group,
                'GroupLabel': GROUP_DISPLAY[group],
                'Model': model,
                'actual_pre': float(row['actual_pre']),
                'actual_post': float(row['actual_post']),
                'pred_pre': float(row['pred_pre']),
                'pred_post': float(row['pred_post']),
                'actual_delta': float(row['actual_delta']),
                'pred_delta': float(row['pred_delta']),
                'delta_error': float(row['delta_error']),
                'abs_delta_error': float(row['abs_delta_error']),
                'sign_match': bool(row['sign_match']),
            })

e3_summary_df = pd.DataFrame(summary_records)
e3_subject_df = pd.DataFrame(subject_records)

if e3_summary_df.empty:
    raise ValueError('No E3 records produced; check input files.')

group_order = {name: idx for idx, name in enumerate(GROUPS)}
model_order = {name: idx for idx, name in enumerate(MODELS)}
e3_summary_df['GroupOrder'] = e3_summary_df['Group'].map(group_order).fillna(999)
e3_summary_df['ModelOrder'] = e3_summary_df['Model'].map(model_order).fillna(999)
e3_summary_df = e3_summary_df.sort_values(['GroupOrder', 'ModelOrder']).reset_index(drop=True)

summary_path = RESULTS_DIR / 'e3_delta_agreement_summary.csv'
subject_path = RESULTS_DIR / 'e3_delta_agreement_subject_level.csv'
e3_summary_df.to_csv(summary_path, index=False)
e3_subject_df.to_csv(subject_path, index=False)

display_cols = [
    'GroupLabel', 'Model', 'NCommonSubjects',
    'MAE_delta', 'RMSE_delta', 'R2_delta',
    'Pearson_r', 'Pearson_p', 'Spearman_rho',
    'SignAgreementPct', 'MeanDeltaError',
    'PairedT_p', 'Wilcoxon_p',
]
display(e3_summary_df[display_cols])
print('Saved:', summary_path)
print('Saved:', subject_path)

In [ ]:
table_full = e3_summary_df[[
    'GroupLabel', 'Model', 'NCommonSubjects',
    'MAE_delta', 'RMSE_delta', 'R2_delta',
    'MeanDeltaError', 'Pearson_r', 'Pearson_p',
    'Spearman_rho', 'Spearman_p',
    'Calibration_slope', 'Calibration_intercept',
    'SignAgreementPct', 'PairedT_p', 'Wilcoxon_p', 'Cohens_dz',
]].rename(columns={
    'GroupLabel': 'Group',
    'NCommonSubjects': 'N',
    'MeanDeltaError': 'Delta bias (pred-obs)',
    'SignAgreementPct': 'Sign agreement (%)',
    'PairedT_p': 'Paired t p',
    'Wilcoxon_p': 'Wilcoxon p',
})

table_full_csv = RESULTS_DIR / 'e3_results_table_full.csv'
table_full_tex = RESULTS_DIR / 'e3_results_table_full.tex'
table_full.to_csv(table_full_csv, index=False)

table_tex_df = table_full.copy()
for col in [
    'MAE_delta', 'RMSE_delta', 'R2_delta', 'Delta bias (pred-obs)',
    'Pearson_r', 'Spearman_rho', 'Calibration_slope', 'Calibration_intercept',
    'Sign agreement (%)', 'Cohens_dz',
]:
    table_tex_df[col] = table_tex_df[col].map(lambda x: round(float(x), 3) if pd.notna(x) else np.nan)

for col in ['Pearson_p', 'Spearman_p', 'Paired t p', 'Wilcoxon p']:
    table_tex_df[col] = table_full[col].map(
        lambda x: '<0.001' if pd.notna(x) and float(x) < 0.001 else (f'{float(x):.3f}' if pd.notna(x) else 'NA')
    )

table_tex_df.to_latex(
    table_full_tex,
    index=False,
    escape=False,
    caption='Experiment 3: agreement between predicted and observed VO2peak change (delta) across all configurations.',
    label='tab:e3_full',
)

best_worst = pd.concat([
    e3_summary_df.sort_values('MAE_delta').head(2),
    e3_summary_df.sort_values('MAE_delta').tail(2),
]).drop_duplicates().reset_index(drop=True)

key_table = best_worst[[
    'GroupLabel', 'Model', 'NCommonSubjects',
    'MAE_delta', 'RMSE_delta', 'R2_delta',
    'Pearson_r', 'Pearson_p',
    'SignAgreementPct',
]].rename(columns={
    'GroupLabel': 'Group',
    'NCommonSubjects': 'N',
    'SignAgreementPct': 'Sign agreement (%)',
})

key_csv = RESULTS_DIR / 'e3_results_table_key_configs.csv'
key_tex = RESULTS_DIR / 'e3_results_table_key_configs.tex'
key_table.to_csv(key_csv, index=False)

key_tex_df = key_table.copy()
for col in ['MAE_delta', 'RMSE_delta', 'R2_delta', 'Pearson_r', 'Sign agreement (%)']:
    key_tex_df[col] = key_tex_df[col].map(lambda x: round(float(x), 3) if pd.notna(x) else np.nan)
key_tex_df['Pearson_p'] = key_table['Pearson_p'].map(
    lambda x: '<0.001' if pd.notna(x) and float(x) < 0.001 else (f'{float(x):.3f}' if pd.notna(x) else 'NA')
)

key_tex_df.to_latex(
    key_tex,
    index=False,
    escape=False,
    caption='Experiment 3: key configurations with lowest and highest delta-prediction MAE.',
    label='tab:e3_key_configs',
)

display(table_tex_df)
print('Saved:', table_full_csv)
print('Saved:', table_full_tex)
print('Saved:', key_csv)
print('Saved:', key_tex)

In [ ]:
# Figure 1: Delta MAE heatmap across configurations
delta_mae_matrix = e3_summary_df.pivot(index='Group', columns='Model', values='MAE_delta').reindex(GROUPS)[list(MODELS)]
delta_mae_vals = delta_mae_matrix.to_numpy(dtype=float)

fig, ax = plt.subplots(figsize=(7.2, 3.4))
im = ax.imshow(delta_mae_vals, cmap='viridis_r', aspect='auto')
ax.set_xticks(np.arange(len(MODELS)), labels=MODELS)
ax.set_yticks(np.arange(len(GROUPS)), labels=[GROUP_DISPLAY[g] for g in GROUPS])
threshold = np.nanmedian(delta_mae_vals)
for i in range(delta_mae_vals.shape[0]):
    for j in range(delta_mae_vals.shape[1]):
        value = delta_mae_vals[i, j]
        txt_color = 'white' if value > threshold else 'black'
        ax.text(j, i, f'{value:.3f}', ha='center', va='center', color=txt_color, fontsize=9)
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Delta MAE [L/min]')
ax.set_title('E3 delta agreement: MAE heatmap')
ax.set_xlabel('Model')
ax.set_ylabel('Feature group')
fig.tight_layout()
fig1_path = IMAGES_DIR / 'e3_delta_mae_heatmap.png'
fig.savefig(fig1_path, dpi=300, bbox_inches='tight')
plt.close(fig)

# Best combo by delta MAE
best_row = e3_summary_df.sort_values('MAE_delta').iloc[0]
best_group = best_row['Group']
best_model = best_row['Model']

best_subject = e3_subject_df[
    (e3_subject_df['Group'] == best_group) & (e3_subject_df['Model'] == best_model)
].copy()

# Figure 2: Predicted vs observed delta for best combo
x = best_subject['actual_delta'].to_numpy(dtype=float)
y = best_subject['pred_delta'].to_numpy(dtype=float)
lims = [float(min(x.min(), y.min())), float(max(x.max(), y.max()))]

fig, ax = plt.subplots(figsize=(5.2, 5.0))
ax.scatter(x, y, color='navy', alpha=0.85)
ax.plot(lims, lims, '--', color='gray', linewidth=1, label='Ideal 1:1')
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel('Observed delta VO2peak [L/min]')
ax.set_ylabel('Predicted delta VO2peak [L/min]')
ax.set_title(f"E3 best combo: {GROUP_DISPLAY[best_group]} + {best_model}")
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend()
fig.tight_layout()
fig2_path = IMAGES_DIR / 'e3_actual_vs_pred_delta_best.png'
fig.savefig(fig2_path, dpi=300, bbox_inches='tight')
plt.close(fig)

# Figure 3: Bland-Altman on delta for best combo
mean_vals = 0.5 * (x + y)
diff_vals = y - x
mean_diff = float(np.mean(diff_vals))
sd_diff = float(np.std(diff_vals, ddof=1)) if len(diff_vals) > 1 else np.nan
loa = 1.96 * sd_diff if pd.notna(sd_diff) else np.nan

fig, ax = plt.subplots(figsize=(6.0, 4.0))
ax.scatter(mean_vals, diff_vals, color='darkorange', alpha=0.85)
ax.axhline(mean_diff, color='black', linestyle='--', linewidth=1, label='Mean bias')
if pd.notna(loa):
    ax.axhline(mean_diff + loa, color='red', linestyle='--', linewidth=0.9, label='Upper LOA')
    ax.axhline(mean_diff - loa, color='red', linestyle='--', linewidth=0.9, label='Lower LOA')
ax.set_xlabel('Mean of observed/predicted delta [L/min]')
ax.set_ylabel('Delta error (pred - obs) [L/min]')
ax.set_title(f"E3 Bland-Altman: {GROUP_DISPLAY[best_group]} + {best_model}")
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend(loc='upper right')
fig.tight_layout()
fig3_path = IMAGES_DIR / 'e3_delta_bland_altman_best.png'
fig.savefig(fig3_path, dpi=300, bbox_inches='tight')
plt.close(fig)

print('Best combo by delta MAE:', GROUP_DISPLAY[best_group], '+', best_model)
print('Saved:', fig1_path)
print('Saved:', fig2_path)
print('Saved:', fig3_path)

In [ ]:
def fmt_p(p: float) -> str:
    if pd.isna(p):
        return 'NA'
    return '<0.001' if float(p) < 0.001 else f'{float(p):.3f}'

e3_sorted = e3_summary_df.sort_values('MAE_delta').reset_index(drop=True)
best_cfg = e3_sorted.iloc[0]
worst_cfg = e3_sorted.iloc[-1]
mean_sign_agreement = float(e3_summary_df['SignAgreementPct'].mean())
n_signif_t = int((e3_summary_df['PairedT_p'] < 0.05).sum())
n_signif_w = int((e3_summary_df['Wilcoxon_p'] < 0.05).sum())

paragraph_1 = (
    f"Experiment 3 assessed agreement between predicted and observed VO2peak change "
    f"($\\Delta\\hat{{y}}$ vs $\\Delta y$) from baseline to follow-up across 12 "
    f"feature-group/model configurations (Table~\\ref{{tab:e3_full}}). "
    f"Delta-prediction MAE ranged from {best_cfg['MAE_delta']:.3f} to {worst_cfg['MAE_delta']:.3f}~L$\\cdot$min$^{{-1}}$."
 )

paragraph_2 = (
    f"The best configuration was {best_cfg['GroupLabel']} + {best_cfg['Model']} "
    f"(MAE$_\\Delta$={best_cfg['MAE_delta']:.3f}, RMSE$_\\Delta$={best_cfg['RMSE_delta']:.3f}, "
    f"$R^2_\\Delta$={best_cfg['R2_delta']:.3f}, Pearson $r$={best_cfg['Pearson_r']:.3f}, "
    f"sign agreement={best_cfg['SignAgreementPct']:.1f}\\%). "
    f"The worst configuration was {worst_cfg['GroupLabel']} + {worst_cfg['Model']} "
    f"(MAE$_\\Delta$={worst_cfg['MAE_delta']:.3f})."
 )

paragraph_3 = (
    f"Across configurations, mean sign agreement was {mean_sign_agreement:.1f}\\%. "
    f"Only {n_signif_t}/12 paired t-tests and {n_signif_w}/12 Wilcoxon tests were significant "
    f"at $p<0.05$, indicating predominantly moderate agreement without consistent systematic "
    f"delta bias at this sample size."
 )

paragraph_4 = (
    "Figure~\\ref{fig:e3_delta_mae_heatmap} shows delta-MAE across all configurations. "
    "Figure~\\ref{fig:e3_scatter_best} and Figure~\\ref{fig:e3_ba_best} show correlation and "
    "Bland--Altman agreement for the best configuration. Table~\\ref{tab:e3_key_configs} "
    "summarizes best/worst configurations."
 )

section_text = "\n\n".join([
    r"\subsection{E3: Agreement of predicted versus observed change}",
    paragraph_1,
    paragraph_2,
    paragraph_3,
    paragraph_4,
]) + "\n"

figures_text = """\\begin{figure}[t]
\\centering
\\includegraphics[width=0.95\\linewidth]{images/e3_delta_mae_heatmap.png}
\\caption{Experiment 3 delta-prediction MAE heatmap across feature groups and models. Lower values indicate better agreement.}
\\label{fig:e3_delta_mae_heatmap}
\\end{figure}

\\begin{figure}[t]
\\centering
\\includegraphics[width=0.95\\linewidth]{images/e3_actual_vs_pred_delta_best.png}
\\caption{Predicted versus observed VO2peak change for the best E3 configuration.}
\\label{fig:e3_scatter_best}
\\end{figure}

\\begin{figure}[t]
\\centering
\\includegraphics[width=0.95\\linewidth]{images/e3_delta_bland_altman_best.png}
\\caption{Bland--Altman analysis for VO2peak change in the best E3 configuration.}
\\label{fig:e3_ba_best}
\\end{figure}
"""

bundle_text = (
    section_text
    + "\n% Auto-generated tables\n"
    + "\\input{results/e3_results_table_full.tex}\n"
    + "\\input{results/e3_results_table_key_configs.tex}\n\n"
    + figures_text
 )

section_path = RESULTS_DIR / 'e3_results_section.tex'
bundle_path = RESULTS_DIR / 'e3_results_with_tables_and_figures.tex'
section_path.write_text(section_text, encoding='utf-8')
bundle_path.write_text(bundle_text, encoding='utf-8')

print(section_text)
print('Saved:', section_path)
print('Saved:', bundle_path)